In [1]:
import os 
import sys
import json
import pickle
from pathlib import Path

from tqdm import tqdm 
import numpy as np 
import torch
import torch.nn.functional as f
from torch.utils.data import dataset, dataloader
from transformers import AutoTokenizer, AutoModel
import datasets
from datasets import load_dataset

import data_utils

In [2]:
server_name = 'ryan'
dataSet = 'textocr' # choose from mmmu, clevr, textocr
dataSlice = 'dev' # choose from dev, val
embedding = 'image' # choose from image, image_text

# Add the inference directory to the PYTHONPATH
if server_name == 'monet':
    som_path = '/home/monet/meitang/cache-of-thoughts/inference'
    dataDir = '/home/monet/meitang/cache-of-thoughts/data'
    os.environ['HF_HOME'] = '/mnt/data/meitang/.cache/huggingface'
elif server_name == 'ryan':
    som_path = '/home/ryan/meitang/cache-of-thoughts-main/inference'
    dataDir = '/home/ryan/meitang/cache-of-thoughts-main/data'
    os.environ['HF_HOME'] = '/home/ryan/.cache/huggingface'
elif server_name == 'meitang':
    som_path = '/Users/17348/Documents/GitHub/cache-of-thoughts/inference'
    dataDir = '/Users/17348/Documents/GitHub/cache-of-thoughts/data'
    os.environ['HF_HOME'] = '/Users/17348/.cache/huggingface'
else:
    pass # modify accordingly

if som_path not in sys.path:
    sys.path.append(som_path)

In [ ]:
data_path = Path(f'../data/{dataSet}')
if data_path not in sys.path:
  	sys.path.append(data_path)

os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + f":{data_path}"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# load dataset
if dataSlice == 'val':
	support_file = os.path.join(dataDir, dataSet, 'support.json')
else:
    support_file = os.path.join(dataDir, dataSet, 'query.json')
with open(support_file, 'r') as f:
    support_meta = json.load(f)
test_dataset = support_meta
print(test_dataset[0])

# load saved conversation with gpt and attach to dataset
gpt_conversation_path = data_path / f'{dataSlice}/{dataSet}_{dataSlice}_gpt4o_response_v2.jsonl' #TODO: actually the test set
gpt_conversations = []
with open(gpt_conversation_path, 'r') as f:
    for line in f:
        # each line is a json
        gpt_conversations.append(line.strip().strip('"'))
data_conversation = datasets.Dataset.from_dict({"conversations": gpt_conversations})
test_dataset = datasets.Dataset.from_list(test_dataset)
test_dataset = datasets.concatenate_datasets([test_dataset, data_conversation], axis=1)

# # load selected hashtags
# selected_hashtags_path = data_path / 'embeddings/single_keyword_embeddings_dict_gte-base-en-v1.5.pkl'
# with open(selected_hashtags_path, 'rb') as f:
#     selected_hashtags = pickle.load(f)
# print(len(selected_hashtags[0]))
# print(len(selected_hashtags[1]))
# print(len(selected_hashtags[2]))

# load keywords
keyword_dir = dataDir.strip('data') + f'keyword/{dataSet}_gpt/{dataSlice}_keyword'
keyword_test_list = data_utils.get_extracted_keywords(keyword_dir)
data_keyword = datasets.Dataset.from_dict({"keywords": keyword_test_list})
test_dataset = datasets.concatenate_datasets([test_dataset, data_keyword], axis=1)
print(len(data_keyword))
print(len(test_dataset))
# test_dataset = datasets.concatenate_datasets([test_dataset, data_keyword], axis=1)


In [5]:
def load_image(img_ids, root_path):
    if isinstance(img_ids, str):
        img_ids = [img_ids]
    images = []
    image_paths = []
    for img_id in img_ids:
        image_path = os.path.join(root_path, img_id)
        image = Image.open(image_path).convert('RGB')
        images.append(image)
        image_paths.append(image_path)
        
    return images, image_paths

In [ ]:
import numpy as np
import torch

print("Torch version:", torch.__version__)


import clip
clip.available_models()

import os
from PIL import Image
import numpy as np
import torch

from collections import defaultdict
import numpy as np
import pickle
from tqdm import tqdm
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

model, preprocess = clip.load("ViT-B/32")
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

print(preprocess)
def mmmu_clip_preprocess_collate_fn(batch):
    images = torch.stack([preprocess(load_image(sample['image'], dataDir)[0][0].convert('RGB')) for sample in batch])
    texts = torch.stack([clip.tokenize(sample['keywords'])[0] for sample in batch])
    return {'image': images, 'keywords':texts}
dataloader = torch.utils.data.DataLoader(test_dataset, 
                                         batch_size=64,
                                         collate_fn=mmmu_clip_preprocess_collate_fn)

print(f'num of data: {len(dataloader)}')
# image = Image.open(img_path).convert('RGB')
# image_input = preprocess(image).unsqueeze(0).cuda()
# with torch.no_grad():
#     image_features = model.encode_image(image_input).float()

clip_features = []
print("here")
with torch.no_grad():
	for batch in tqdm(dataloader):
		images = batch['image'].to('cuda')
		texts = batch['keywords'].to('cuda')
	    # text_inputs = clip.tokenize(batch['conversation'], truncate=True).to('cuda')
		image_features = model.encode_image(images).float()
		text_features = model.encode_text(texts).float()
        # text_features = model.encode_text(text_inputs).float()
        #logits_per_image, logits_per_text = model(images.to('cuda'), text_inputs)
        #probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        #print(image_features)
        #print(probs)
        #text_features = model.encode_text(labels)
        #logits_per_image, logits_per_text = model(image, text)
        #probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        #print(probs)
	    #clip_features.append[[image_features,text_features]]
		if embedding == 'image':
			clip_features.append(image_features.cpu())
		else:
			clip_features.append((image_features.cpu()+text_features.cpu())/2)
print(len(clip_features))
print(clip_features[0])
#filename = 'mean_features.pkl'
#with open(filename, 'wb') as file:
#    pickle.dump(mean_features, file)

In [ ]:
clip_features_concat = torch.cat(clip_features, dim=0)
clip_features_concat.shape

In [8]:
clip_features_concat = torch.cat(clip_features, dim=0)
clip_embeddings_file = data_path / f'{dataSlice}/clip/{dataSet}_{dataSlice}_{embedding}_clip_embd_cold_start.pkl'
with open(clip_embeddings_file, 'wb') as f:
    pickle.dump(clip_features_concat, f)